# JOCAS -> SIREN : expérimentation silver labels

Ce notebook teste une approche de *distant supervision* : récupérer un SIREN manquant quand le même nom d'entreprise apparaît ailleurs dans JOCAS avec un SIREN renseigné.

Ces labels sont des **silver labels**, pas un gold standard. Le notebook force donc trois contrôles : filtre géographique, diagnostic d'homonymie, et audit manuel d'un échantillon.

In [1]:
from __future__ import annotations

import json
import math
import os
import random
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, precision_recall_curve, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

try:
    import lightgbm as lgb
except ImportError:
    lgb = None

try:
    from rapidfuzz import fuzz, distance
except ImportError:
    fuzz = None
    distance = None

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "jocas_common.py").exists():
            REPO_ROOT = parent
            break
sys.path.insert(0, str(REPO_ROOT / "src"))

from jocas_common import JOCAS_S3_GLOB, sql_clean_numeric, sql_is_blacklisted_name, sql_name_norm, sql_siren_norm

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

WORK_DIR = REPO_ROOT / "jocas_silver_work"
WORK_DIR.mkdir(exist_ok=True)
print("REPO_ROOT =", REPO_ROOT)
print("WORK_DIR  =", WORK_DIR)

ModuleNotFoundError: No module named 'sentence_transformers'

## 1. Paramètres

`STRICT_GEO_LEVEL="department"` est le point de départ recommandé. Passer à `zipcode` ou `city_zipcode` si l'audit révèle trop d'homonymes.

In [ ]:
JOCAS_SOURCE_MODE = "s3"  # "s3" ou "local_glob"
JOCAS_LOCAL_GLOB = str(REPO_ROOT / "data" / "jocas_sample" / "*.parquet")

STRICT_GEO_LEVEL = "department"  # "department", "zipcode", "city_zipcode"
MIN_CANON_NAME_LEN = 3
DROP_BLACKLISTED_NAMES = True
REQUIRE_UNAMBIGUOUS_NAME_GEO = True

MAX_POSITIVE_PAIRS = 2_000_000
MAX_NEGATIVE_PAIRS = 2_000_000
AUDIT_SAMPLE_SIZE = 1_000

POSITIVE_PAIRS_PATH = WORK_DIR / f"silver_positive_pairs_{STRICT_GEO_LEVEL}.parquet"
NEGATIVE_PAIRS_PATH = WORK_DIR / f"silver_negative_pairs_{STRICT_GEO_LEVEL}.parquet"
TRAINING_PAIRS_PATH = WORK_DIR / f"silver_training_pairs_{STRICT_GEO_LEVEL}.parquet"
AUDIT_SAMPLE_PATH = WORK_DIR / f"manual_audit_sample_{STRICT_GEO_LEVEL}.csv"
ERROR_AUDIT_PATH = WORK_DIR / f"silver_error_audit_{STRICT_GEO_LEVEL}.csv"
MODEL_PATH = WORK_DIR / f"lightgbm_jocas_silver_{STRICT_GEO_LEVEL}.txt"
SUMMARY_PATH = WORK_DIR / f"experiment_summary_{STRICT_GEO_LEVEL}.json"

## 2. Connexion DuckDB à JOCAS

La vue attend les colonnes `entreprise_nom`, `entreprise_siren`, `location_label`, `location_zipcode`, `location_departement`.

In [ ]:
def configure_s3_if_available(conn: duckdb.DuckDBPyConnection) -> None:
    try:
        conn.execute("INSTALL httpfs; LOAD httpfs;")
    except Exception as exc:
        print(f"Attention: httpfs non chargé ({type(exc).__name__}: {exc})")
    settings = {
        "s3_endpoint": os.getenv("AWS_S3_ENDPOINT", "minio.lab.sspcloud.fr"),
        "s3_url_style": os.getenv("AWS_S3_URL_STYLE", "path"),
        "s3_region": os.getenv("AWS_DEFAULT_REGION", "us-east-1"),
        "s3_access_key_id": os.getenv("AWS_ACCESS_KEY_ID"),
        "s3_secret_access_key": os.getenv("AWS_SECRET_ACCESS_KEY"),
        "s3_session_token": os.getenv("AWS_SESSION_TOKEN"),
    }
    for key, value in settings.items():
        if value:
            conn.execute(f"SET {key}=?", [value])

def create_jocas_view(conn: duckdb.DuckDBPyConnection) -> None:
    if JOCAS_SOURCE_MODE == "s3":
        configure_s3_if_available(conn)
        source = JOCAS_S3_GLOB
    elif JOCAS_SOURCE_MODE == "local_glob":
        source = JOCAS_LOCAL_GLOB.replace("\\", "/")
    else:
        raise ValueError("JOCAS_SOURCE_MODE doit valoir 's3' ou 'local_glob'.")
    conn.execute("DROP VIEW IF EXISTS jocas")
    conn.execute("""
        CREATE VIEW jocas AS
        SELECT *
        FROM read_parquet(?, hive_partitioning=true, union_by_name=true)
    """, [source])

conn = duckdb.connect(database=":memory:")
create_jocas_view(conn)
schema = conn.sql("DESCRIBE jocas").df()
display(schema)
required = {"entreprise_nom", "entreprise_siren", "location_label", "location_zipcode", "location_departement"}
missing = required - set(schema["column_name"].str.lower())
if missing:
    raise RuntimeError(f"Colonnes JOCAS manquantes: {sorted(missing)}")
conn.sql("SELECT * FROM jocas LIMIT 5").df()

## 3. Vue normalisée, couverture et ambiguïtés

On réutilise les helpers SQL du repo pour normaliser de la même façon que le pipeline existant.

In [ ]:
blacklist_predicate = sql_is_blacklisted_name("entreprise_nom") if DROP_BLACKLISTED_NAMES else "false"
conn.execute("DROP VIEW IF EXISTS jocas_norm")
conn.execute(f"""
CREATE VIEW jocas_norm AS
SELECT
    row_number() OVER () AS row_id,
    entreprise_nom,
    {sql_name_norm('entreprise_nom')} AS nom_key,
    {sql_siren_norm('entreprise_siren')} AS siren_norm,
    location_label,
    {sql_name_norm('location_label')} AS city_key,
    {sql_clean_numeric('location_zipcode')} AS zipcode_norm,
    {sql_clean_numeric('location_departement')} AS dept_norm,
    {blacklist_predicate} AS is_blacklisted
FROM jocas
WHERE entreprise_nom IS NOT NULL
  AND trim(entreprise_nom) != ''
  AND length({sql_name_norm('entreprise_nom')}) >= {MIN_CANON_NAME_LEN}
""")

display(conn.sql("""
SELECT count(*) AS n_rows_with_name,
       count(*) FILTER (WHERE siren_norm IS NOT NULL) AS n_with_siren,
       count(*) FILTER (WHERE siren_norm IS NULL) AS n_without_siren,
       count(*) FILTER (WHERE is_blacklisted) AS n_blacklisted,
       count(DISTINCT nom_key) AS n_distinct_names
FROM jocas_norm
""").df())

def ambiguity_report(group_cols: list[str], label: str) -> pd.DataFrame:
    cols = ", ".join(group_cols)
    return conn.sql(f"""
        WITH grouped AS (
            SELECT {cols}, count(*) AS n_rows,
                   count(DISTINCT siren_norm) FILTER (WHERE siren_norm IS NOT NULL) AS n_sirens
            FROM jocas_norm
            WHERE NOT is_blacklisted
            GROUP BY {cols}
        )
        SELECT '{label}' AS grain,
               count(*) AS n_groups,
               count(*) FILTER (WHERE n_sirens = 1) AS n_unambiguous_groups,
               count(*) FILTER (WHERE n_sirens > 1) AS n_ambiguous_groups,
               sum(n_rows) AS n_rows,
               sum(n_rows) FILTER (WHERE n_sirens > 1) AS rows_ambiguous
        FROM grouped
    """).df()

display(pd.concat([
    ambiguity_report(["nom_key"], "name"),
    ambiguity_report(["nom_key", "dept_norm"], "name+department"),
    ambiguity_report(["nom_key", "zipcode_norm"], "name+zipcode"),
    ambiguity_report(["nom_key", "city_key", "zipcode_norm"], "name+city+zipcode"),
], ignore_index=True))

conn.sql("""
SELECT nom_key, count(*) AS n_rows,
       count(DISTINCT siren_norm) FILTER (WHERE siren_norm IS NOT NULL) AS n_sirens,
       string_agg(DISTINCT siren_norm, ', ' ORDER BY siren_norm) FILTER (WHERE siren_norm IS NOT NULL) AS sirens
FROM jocas_norm
WHERE NOT is_blacklisted
GROUP BY nom_key
HAVING count(DISTINCT siren_norm) FILTER (WHERE siren_norm IS NOT NULL) > 1
ORDER BY n_rows DESC, n_sirens DESC
LIMIT 50
""").df()

## 4. Paires positives silver et audit manuel

Une positive = offre sans SIREN + SIREN connu ailleurs, même `nom_key`, géographie cohérente, et groupe nom+géo non ambigu par défaut.

In [ ]:
def geo_join_condition(level: str, left_alias: str = "m", right_alias: str = "k") -> str:
    if level == "department":
        return f"coalesce({left_alias}.dept_norm, '') = coalesce({right_alias}.dept_norm, '') AND coalesce({left_alias}.dept_norm, '') != ''"
    if level == "zipcode":
        return f"coalesce({left_alias}.zipcode_norm, '') = coalesce({right_alias}.zipcode_norm, '') AND coalesce({left_alias}.zipcode_norm, '') != ''"
    if level == "city_zipcode":
        return f"coalesce({left_alias}.zipcode_norm, '') = coalesce({right_alias}.zipcode_norm, '') AND coalesce({left_alias}.city_key, '') = coalesce({right_alias}.city_key, '') AND coalesce({left_alias}.zipcode_norm, '') != ''"
    raise ValueError(level)

GEO_CONDITION = geo_join_condition(STRICT_GEO_LEVEL)
GEO_GROUP_COLS = {"department": ["nom_key", "dept_norm"], "zipcode": ["nom_key", "zipcode_norm"], "city_zipcode": ["nom_key", "city_key", "zipcode_norm"]}[STRICT_GEO_LEVEL]
GEO_GROUP_SQL = ", ".join(GEO_GROUP_COLS)
GEO_GROUP_JOIN = " AND ".join([f"m.{c} IS NOT DISTINCT FROM u.{c}" for c in GEO_GROUP_COLS])
positive_limit = "" if MAX_POSITIVE_PAIRS is None else f"LIMIT {int(MAX_POSITIVE_PAIRS)}"

positive_sql = f"""
WITH known AS (
    SELECT nom_key, siren_norm AS target_siren,
           any_value(entreprise_nom) AS known_entreprise_nom,
           any_value(location_label) AS known_location_label,
           city_key, zipcode_norm, dept_norm, count(*) AS n_known_rows
    FROM jocas_norm
    WHERE siren_norm IS NOT NULL AND NOT is_blacklisted
    GROUP BY nom_key, target_siren, city_key, zipcode_norm, dept_norm
),
missing AS (
    SELECT row_id AS missing_row_id, entreprise_nom AS missing_entreprise_nom,
           nom_key, location_label AS missing_location_label, city_key, zipcode_norm, dept_norm
    FROM jocas_norm
    WHERE siren_norm IS NULL AND NOT is_blacklisted
),
unambiguous AS (
    SELECT {GEO_GROUP_SQL}, count(DISTINCT siren_norm) AS n_sirens_at_grain
    FROM jocas_norm
    WHERE siren_norm IS NOT NULL AND NOT is_blacklisted
    GROUP BY {GEO_GROUP_SQL}
)
SELECT m.missing_row_id, m.missing_entreprise_nom, m.missing_location_label,
       m.nom_key, m.city_key, m.zipcode_norm, m.dept_norm,
       k.target_siren, k.known_entreprise_nom, k.known_location_label,
       k.n_known_rows, u.n_sirens_at_grain, 1 AS label
FROM missing m
JOIN known k ON m.nom_key = k.nom_key AND {GEO_CONDITION}
JOIN unambiguous u ON {GEO_GROUP_JOIN}
WHERE {'u.n_sirens_at_grain = 1' if REQUIRE_UNAMBIGUOUS_NAME_GEO else 'true'}
{positive_limit}
"""
conn.execute(f"COPY ({positive_sql}) TO ? (FORMAT PARQUET)", [str(POSITIVE_PAIRS_PATH)])
display(conn.sql("""
SELECT count(*) AS n_positive_pairs,
       count(DISTINCT missing_row_id) AS n_missing_rows_covered,
       count(DISTINCT nom_key) AS n_names_covered,
       count(DISTINCT target_siren) AS n_target_sirens
FROM read_parquet(?)
""", params=[str(POSITIVE_PAIRS_PATH)]).df())

audit_df = conn.sql(f"""
SELECT missing_row_id, missing_entreprise_nom, missing_location_label,
       zipcode_norm, dept_norm, target_siren, known_entreprise_nom,
       known_location_label, n_known_rows, n_sirens_at_grain,
       '' AS audit_decision, '' AS audit_comment
FROM read_parquet(?)
ORDER BY random()
LIMIT {AUDIT_SAMPLE_SIZE}
""", params=[str(POSITIVE_PAIRS_PATH)]).df()
audit_df.to_csv(AUDIT_SAMPLE_PATH, index=False, encoding="utf-8")
print("Audit sample:", AUDIT_SAMPLE_PATH)
display(audit_df.head(20))

## 5. Estimer le bruit après audit

Après avoir rempli `audit_decision` dans le CSV (`1`, `0`, ou `?`), relancer cette cellule.

In [ ]:
reviewed = pd.read_csv(AUDIT_SAMPLE_PATH)
decisions = reviewed["audit_decision"].astype(str).str.strip()
reviewed = reviewed[decisions.isin(["0", "1"])].copy()
if len(reviewed):
    y = reviewed["audit_decision"].astype(int)
    noise = 1 - y.mean()
    ci95 = 1.96 * math.sqrt(noise * (1 - noise) / len(y))
    print(f"Cas audités: {len(y)} ; bruit estimé: {noise:.2%} ± {ci95:.2%}")
else:
    print("Aucune décision 0/1 renseignée pour l'instant.")

## 6. Paires négatives

On crée des négatifs dans la même zone géographique mais avec un autre nom/SIREN, plus des négatifs aléatoires.

In [ ]:
pos_path_sql = str(POSITIVE_PAIRS_PATH).replace("\\", "/")
negative_limit_each = max(1, int(MAX_NEGATIVE_PAIRS // 2))
negative_sql = f"""
WITH positives AS (SELECT * FROM read_parquet('{pos_path_sql}')),
known_by_geo AS (
    SELECT nom_key, siren_norm AS candidate_siren,
           any_value(entreprise_nom) AS candidate_name,
           any_value(location_label) AS candidate_location_label,
           city_key, zipcode_norm, dept_norm, count(*) AS n_candidate_rows
    FROM jocas_norm
    WHERE siren_norm IS NOT NULL AND NOT is_blacklisted
    GROUP BY nom_key, candidate_siren, city_key, zipcode_norm, dept_norm
),
hard_geo AS (
    SELECT p.missing_row_id, p.missing_entreprise_nom, p.missing_location_label,
           p.nom_key, p.city_key, p.zipcode_norm, p.dept_norm,
           k.candidate_siren AS target_siren, k.candidate_name AS known_entreprise_nom,
           k.candidate_location_label AS known_location_label, k.n_candidate_rows AS n_known_rows,
           0 AS n_sirens_at_grain, 0 AS label, 'hard_geo' AS negative_type
    FROM positives p
    JOIN known_by_geo k ON {geo_join_condition(STRICT_GEO_LEVEL, 'p', 'k')}
       AND p.nom_key != k.nom_key AND p.target_siren != k.candidate_siren
    USING SAMPLE reservoir({negative_limit_each} ROWS) REPEATABLE ({RANDOM_SEED})
),
random_neg AS (
    SELECT p.missing_row_id, p.missing_entreprise_nom, p.missing_location_label,
           p.nom_key, p.city_key, p.zipcode_norm, p.dept_norm,
           k.candidate_siren AS target_siren, k.candidate_name AS known_entreprise_nom,
           k.candidate_location_label AS known_location_label, k.n_candidate_rows AS n_known_rows,
           0 AS n_sirens_at_grain, 0 AS label, 'random' AS negative_type
    FROM positives p
    JOIN known_by_geo k ON p.target_siren != k.candidate_siren AND p.nom_key != k.nom_key
    USING SAMPLE reservoir({negative_limit_each} ROWS) REPEATABLE ({RANDOM_SEED + 1})
)
SELECT * FROM hard_geo
UNION ALL
SELECT * FROM random_neg
LIMIT {MAX_NEGATIVE_PAIRS}
"""
conn.execute(f"COPY ({negative_sql}) TO ? (FORMAT PARQUET)", [str(NEGATIVE_PAIRS_PATH)])
conn.sql("SELECT label, negative_type, count(*) AS n FROM read_parquet(?) GROUP BY 1, 2 ORDER BY 1, 2", params=[str(NEGATIVE_PAIRS_PATH)]).df()

## 7. Features, entraînement LightGBM et analyse d'erreurs

Le split est groupé par `nom_key` pour éviter une validation trop optimiste.

In [ ]:
if lgb is None:
    raise ImportError("lightgbm n'est pas installé dans cet environnement.")

neg_path_sql = str(NEGATIVE_PAIRS_PATH).replace("\\", "/")
conn.execute(f"""
COPY (
    SELECT *, 'positive' AS negative_type FROM read_parquet('{pos_path_sql}')
    UNION ALL
    SELECT * FROM read_parquet('{neg_path_sql}')
) TO ? (FORMAT PARQUET)
""", [str(TRAINING_PAIRS_PATH)])
pairs = pd.read_parquet(TRAINING_PAIRS_PATH)
print(pairs.shape)
display(pairs["label"].value_counts(dropna=False))

def safe_text(x) -> str:
    return "" if pd.isna(x) else str(x)

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    left = out["missing_entreprise_nom"].map(safe_text).str.lower()
    right = out["known_entreprise_nom"].map(safe_text).str.lower()
    left_key = out["nom_key"].map(safe_text).str.lower()
    right_key = right.str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii")
    if fuzz is not None:
        out["fuzz_ratio"] = [fuzz.ratio(a, b) / 100 for a, b in zip(left, right)]
        out["fuzz_token_sort"] = [fuzz.token_sort_ratio(a, b) / 100 for a, b in zip(left, right)]
        out["fuzz_token_set"] = [fuzz.token_set_ratio(a, b) / 100 for a, b in zip(left, right)]
        out["partial_ratio"] = [fuzz.partial_ratio(a, b) / 100 for a, b in zip(left, right)]
        out["levenshtein_norm"] = [distance.Levenshtein.normalized_similarity(a, b) for a, b in zip(left, right)]
        out["jaro_winkler"] = [distance.JaroWinkler.similarity(a, b) for a, b in zip(left, right)]
    else:
        from difflib import SequenceMatcher
        sim = [SequenceMatcher(None, a, b).ratio() for a, b in zip(left, right)]
        out["fuzz_ratio"] = sim; out["fuzz_token_sort"] = sim; out["fuzz_token_set"] = sim
        out["partial_ratio"] = sim; out["levenshtein_norm"] = sim; out["jaro_winkler"] = sim
    out["name_exact_key"] = (left_key == right_key).astype(int)
    out["has_dept"] = out["dept_norm"].notna().astype(int)
    out["has_zipcode"] = out["zipcode_norm"].notna().astype(int)
    out["has_city"] = out["city_key"].notna().astype(int)
    out["log_known_rows"] = np.log1p(out["n_known_rows"].fillna(0).astype(float))
    out["left_len"] = left.str.len().fillna(0)
    out["right_len"] = right.str.len().fillna(0)
    out["len_ratio"] = (out["left_len"].clip(lower=1) / out["right_len"].clip(lower=1)).clip(0, 5)
    return out

pairs_feat = add_features(pairs)
feature_cols = ["fuzz_ratio", "fuzz_token_sort", "fuzz_token_set", "partial_ratio", "levenshtein_norm", "jaro_winkler", "name_exact_key", "has_dept", "has_zipcode", "has_city", "log_known_rows", "left_len", "right_len", "len_ratio"]
model_df = pairs_feat.dropna(subset=["label"]).copy()
model_df[feature_cols] = model_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
model_df["label"] = model_df["label"].astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, valid_idx = next(splitter.split(model_df, model_df["label"], groups=model_df["nom_key"]))
train_df = model_df.iloc[train_idx].copy()
valid_df = model_df.iloc[valid_idx].copy()

train_set = lgb.Dataset(train_df[feature_cols], label=train_df["label"], feature_name=feature_cols)
valid_set = lgb.Dataset(valid_df[feature_cols], label=valid_df["label"], feature_name=feature_cols, reference=train_set)
params = {"objective":"binary", "metric":["auc", "average_precision", "binary_logloss"], "learning_rate":0.05, "num_leaves":31, "min_data_in_leaf":100, "feature_fraction":0.9, "bagging_fraction":0.9, "bagging_freq":1, "verbosity":-1, "seed":RANDOM_SEED, "is_unbalance":True}
model = lgb.train(params, train_set, valid_sets=[train_set, valid_set], valid_names=["train", "valid"], num_boost_round=500, callbacks=[lgb.early_stopping(50), lgb.log_evaluation(25)])
model.save_model(str(MODEL_PATH))

valid_pred = model.predict(valid_df[feature_cols], num_iteration=model.best_iteration)
threshold = 0.5
print("ROC AUC:", roc_auc_score(valid_df["label"], valid_pred))
print("Average precision:", average_precision_score(valid_df["label"], valid_pred))
print(classification_report(valid_df["label"], valid_pred >= threshold, digits=4))
display(pd.DataFrame(confusion_matrix(valid_df["label"], valid_pred >= threshold), index=["true_0", "true_1"], columns=["pred_0", "pred_1"]))

display(pd.DataFrame({"feature": feature_cols, "gain": model.feature_importance(importance_type="gain"), "split": model.feature_importance(importance_type="split")}).sort_values("gain", ascending=False))

valid_errors = valid_df.copy()
valid_errors["pred"] = valid_pred
valid_errors["pred_label"] = (valid_errors["pred"] >= threshold).astype(int)
valid_errors["error_type"] = np.select([(valid_errors.label == 1) & (valid_errors.pred_label == 0), (valid_errors.label == 0) & (valid_errors.pred_label == 1)], ["silver_positive_low_score", "negative_high_score"], default="ok")
cols = ["error_type", "pred", "label", "missing_entreprise_nom", "known_entreprise_nom", "missing_location_label", "known_location_label", "zipcode_norm", "dept_norm", "target_siren", "negative_type"]
errors_to_audit = valid_errors.query("error_type != 'ok'").sort_values("pred", ascending=False)[cols]
errors_to_audit.head(2000).to_csv(ERROR_AUDIT_PATH, index=False, encoding="utf-8")
display(errors_to_audit.head(50))

## 8. Gold set optionnel et résumé

Créer `jocas_silver_work/gold_pairs.csv` avec `missing_entreprise_nom`, `known_entreprise_nom` ou `candidate_name`, `target_siren` ou `candidate_siren`, `label`. Ce fichier doit rester hors entraînement.

In [ ]:
GOLD_PATH = WORK_DIR / "gold_pairs.csv"
if GOLD_PATH.exists():
    gold = pd.read_parquet(GOLD_PATH) if GOLD_PATH.suffix.lower() == ".parquet" else pd.read_csv(GOLD_PATH)
    gold = gold.rename(columns={"candidate_name": "known_entreprise_nom", "candidate_siren": "target_siren"})
    for col, default in {"missing_location_label": None, "known_location_label": None, "city_key": None, "zipcode_norm": None, "dept_norm": None, "n_known_rows": 1, "nom_key": None}.items():
        if col not in gold.columns:
            gold[col] = default
    if gold["nom_key"].isna().all():
        gold["nom_key"] = gold["missing_entreprise_nom"].map(safe_text).str.lower()
    gold_feat = add_features(gold)
    gold_feat[feature_cols] = gold_feat[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    gold_y = gold_feat["label"].astype(int)
    gold_pred = model.predict(gold_feat[feature_cols], num_iteration=model.best_iteration)
    print("Gold ROC AUC:", roc_auc_score(gold_y, gold_pred))
    print("Gold Average precision:", average_precision_score(gold_y, gold_pred))
    print(classification_report(gold_y, gold_pred >= threshold, digits=4))
else:
    print(f"Pas de gold set trouvé: {GOLD_PATH}")

summary = {
    "strict_geo_level": STRICT_GEO_LEVEL,
    "positive_pairs_path": str(POSITIVE_PAIRS_PATH),
    "negative_pairs_path": str(NEGATIVE_PAIRS_PATH),
    "training_pairs_path": str(TRAINING_PAIRS_PATH),
    "audit_sample_path": str(AUDIT_SAMPLE_PATH),
    "error_audit_path": str(ERROR_AUDIT_PATH),
    "model_path": str(MODEL_PATH),
    "n_pairs": int(len(pairs_feat)),
    "positive_rate": float(pairs_feat["label"].mean()),
    "valid_roc_auc_silver": float(roc_auc_score(valid_df["label"], valid_pred)),
    "valid_average_precision_silver": float(average_precision_score(valid_df["label"], valid_pred)),
    "best_iteration": int(model.best_iteration or 0),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
summary